# Electricity Demand Forecasting in Tamil Nadu
Ayush Kumar Jha (BMC202416) and Rohan Roy (BMC202438)  
Time Series Analysis · Professor M. R Srinivasan · November 2026

This notebook reproduces the cleaned daily series, model grid, forecast errors and the figures used in the report. Run from the extracted project directory. Cached table previews below come from the included result files.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from analysis import run
from figures import draw_figures
from report_tables import make_tables
root = Path.cwd()
results = root / 'results'
def table(stem):
    return pd.read_csv(results / f'{stem}.csv')


## 1. Reproduce the analysis
Use the three Tamil Nadu extracts in data/. Dates are merged on a regular calendar. Gaps remain blank for descriptive and scoring purposes. The analysis also takes the original three CSVs under their supplied names if they are placed in data/.

In [ ]:
summary = run(root / 'data', results)
draw_figures(results, root / 'figures')
make_tables(results)
display(pd.DataFrame(summary['splits']).T)

In [ ]:
display(table('tamil_nadu_clean_daily').head(8))

date,peak_mw,energy_mu,temp_mean_c,weekday,season,split
2013-01-01,11924,245.3,25.948,Tuesday,Winter,train
2013-01-02,11816,247.7,25.377,Wednesday,Winter,train
2013-01-03,11887,253.8,25.74,Thursday,Winter,train
2013-01-04,12111,252.2,25.953,Friday,Winter,train
2013-01-05,11311,243.9,26.148,Saturday,Winter,train
2013-01-06,11737,254.5,26.358,Sunday,Winter,train
2013-01-07,12635,262.3,25.633,Monday,Winter,train
2013-01-08,NaN,NaN,25.082,Tuesday,Winter,train


## 2. Descriptive analysis and stationarity
Figures 1–7 cover trends, seasonal patterns and temperature. Figures 8–9 show training ACF and PACF values. The ADF uses a common set of complete lag rows per transformation and the intercept-only MacKinnon 5% critical value.

In [ ]:
display(pd.DataFrame(summary['correlations']).T)
display(table('adf'))

series,transformation,statistic,lag,n,critical_5,reject
peak_mw,Level,-4.4045,14,1889,-2.8631,True
peak_mw,First difference,-15.855,13,1853,-2.8631,True
peak_mw,Weekly difference,-11.669,14,1679,-2.8633,True
peak_mw,First & weekly,-19.767,14,1652,-2.8633,True
energy_mu,Level,-4.8333,14,1889,-2.8631,True
energy_mu,First difference,-13.844,13,1853,-2.8631,True
energy_mu,Weekly difference,-10.148,14,1679,-2.8633,True
energy_mu,First & weekly,-21.715,13,1652,-2.8633,True
temp_mean_c,Level,-4.3375,7,2878,-2.8625,True
temp_mean_c,First difference,-26.561,6,2877,-2.8625,True


In [ ]:
display(table('demand_correlograms').query('lag in [1, 7, 14]').reset_index(drop=True))

## 3. Fit and compare five forecasting models
Compare candidates on validation blocks and refit the selected specifications before testing. The candidate table lists conditional AIC and BIC alongside validation errors. The final table reports matched-date MAE and RMSE.

In [ ]:
grid = table('candidate_grid')
display(grid.sort_values(['target','family','rmse']).groupby(['target','family']).head(3))

target,family,spec,aic,bic,mae,rmse
peak_mw,ARIMA,"(0,1,0)",41797,41803,748.87,1017.7
peak_mw,ARIMA,"(1,1,0)",41626,41638,752.83,1017.1
peak_mw,ARIMA,"(0,1,1)",41495,41507,758.57,990.43
peak_mw,ARIMA,"(1,1,1)",41283,41301,796.96,1012.4
peak_mw,ARIMA,"(2,1,1)",41285,41308,797.39,1012.5
peak_mw,ARIMA,"(1,1,2)",41285,41308,797.42,1012.6
peak_mw,ARIMA,"(1,0,1)",42734,42758,792.15,985.86
peak_mw,ARIMA,"(2,0,1)",42608,42637,820.61,1008.7


In [ ]:
comparison = table('model_comparison')
columns = ['target','model','spec','val_mae','val_rmse','test_mae','test_rmse']
display(comparison[columns])

target,model,spec,val_mae,val_rmse,test_mae,test_rmse
peak_mw,Seasonal naive,Weekly lag 7,763.46,1023,856.22,1146.6
peak_mw,ARIMA,"(1,0,1)",792.15,985.86,1297.2,1490.1
peak_mw,SARIMA,"(0,1,1)(0,1,1)[7]",613.51,786.94,705.38,964.05
peak_mw,Holt-Winters,"Additive, damped",608.32,777.8,690.32,952.95
peak_mw,SARIMAX,"(0,1,1)(0,1,1)[7] + temperature",594.91,775.54,683.72,951.3
energy_mu,Seasonal naive,Weekly lag 7,18.132,24.035,20.209,26.396
energy_mu,ARIMA,"(1,0,1)",16.739,21.416,32.101,36.565
energy_mu,SARIMA,"(1,1,1)(0,1,1)[7]",15.051,18.859,14.805,19.528
energy_mu,Holt-Winters,"Additive, damped",14.839,18.818,14.581,20.39
energy_mu,SARIMAX,"(1,1,1)(0,1,1)[7] + temperature",14.555,18.681,14.255,19.133


In [ ]:
display(table('residual_summary'))

target,model,n,mean,sd,acf_1,acf_7,acf_14
peak_mw,Seasonal naive,3140,0.3949,937.15,0.69486,-0.34678,-0.044743
peak_mw,ARIMA,3273,-0.040923,756.52,0.11955,0.50551,0.51633
peak_mw,SARIMA,2692,0.27661,533.38,0.033507,0.016706,0.031558
peak_mw,Holt-Winters,3273,0.67376,539.26,0.090013,0.033325,0.036415
peak_mw,SARIMAX,2692,-2.7079,509.16,0.043165,0.020145,0.035685
energy_mu,Seasonal naive,3140,0.028153,21.804,0.80635,-0.31284,-0.033762
energy_mu,ARIMA,3273,-0.00017205,13.458,0.065852,0.46411,0.45721
energy_mu,SARIMA,2692,0.010052,9.7814,0.11451,0.042691,0.052266
energy_mu,Holt-Winters,3273,0.019278,10.507,0.12273,0.041301,0.031442
energy_mu,SARIMAX,2692,-0.082971,9.2295,0.083414,0.052111,0.051101


## 4. Temperature and predictive direction
The Granger tests use changes with weekday and annual controls. SARIMAX consumes independent, recursively generated temperature forecasts for every test block. A paired 28-day block bootstrap summarizes uncertainty in the matched RMSE differences.

In [ ]:
display(table('granger'))
display(pd.DataFrame(summary['paired_sarimax_minus_sarima']).T)

target,direction,f,df1,df2,p,wald,p_hac,n,holm_hac
peak_mw,temperature_to_demand,9.6946,6,1866,1.5589e-10,39.802,4.9828e-07,1889,1.4948e-06
peak_mw,demand_to_temperature,0.8292,6,1866,0.54716,5.3412,0.50085,1889,0.50085
energy_mu,temperature_to_demand,14.721,8,1862,4.7809e-21,72.765,1.3808e-12,1889,5.5232e-12
energy_mu,demand_to_temperature,2.8946,8,1862,0.0032997,23.442,0.0028406,1889,0.0056812


In [ ]:
weather = summary['temperature']
display(pd.DataFrame({k: weather[k] for k in ['validation','test','test_persistence']}).T[['mae','rmse']])
display(table('temperature_test').head(7))

date,actual_c,predicted_c,horizon
2022-08-15,28.812,28.216,1
2022-08-16,28.856,28.233,2
2022-08-17,28.509,28.249,3
2022-08-18,28.383,28.25,4
2022-08-19,28.405,28.259,5
2022-08-20,28.199,28.271,6
2022-08-21,28.239,28.27,7


## 5. Test forecasts and errors
The saved daily forecast table contains all five models for each target. Error profiles by lead and season use the same scored dates. The PDF figures are vector graphics.

In [ ]:
display(table('forecasts_test').head(10))

date,target,actual,horizon,season,Seasonal naive,ARIMA,SARIMA,Holt-Winters,SARIMAX
2022-08-15,peak_mw,13659,1,SW monsoon,14435,14687,15615,15664,15586
2022-08-16,peak_mw,16058,2,SW monsoon,14933,14634,15732,15791,15715
2022-08-17,peak_mw,15932,3,SW monsoon,15278,14582,15725,15816,15736
2022-08-18,peak_mw,15982,4,SW monsoon,15883,14533,15788,15890,15801
2022-08-19,peak_mw,15906,5,SW monsoon,15941,14485,15821,15873,15791
2022-08-20,peak_mw,15550,6,SW monsoon,15780,14438,15597,15590,15565
2022-08-21,peak_mw,13840,7,SW monsoon,13920,14393,14001,13984,13980
2022-08-22,peak_mw,14937,1,SW monsoon,13659,14637,15434,15498,15451
2022-08-23,peak_mw,15506,2,SW monsoon,16058,14585,15644,15752,15673
2022-08-24,peak_mw,15249,3,SW monsoon,15932,14535,15633,15731,15686


In [ ]:
display(table('seasonal_errors').query('model == "SARIMAX"'))
display(table('horizon_errors').query('model == "SARIMAX"'))
display(table('largest_test_errors'))

target,model,date,actual,forecast,error,horizon
peak_mw,SARIMAX,2023-05-07,12621,17263,4641.9,7
peak_mw,SARIMAX,2023-05-06,14421,19050,4628.7,6
peak_mw,SARIMAX,2023-05-05,14562,19146,4584.5,5
peak_mw,SARIMAX,2023-05-01,13897,18457,4559.5,1
peak_mw,SARIMAX,2023-05-04,14614,19108,4494.1,4
energy_mu,SARIMAX,2023-05-02,299,379.81,80.806,2
energy_mu,SARIMAX,2023-05-06,307,384.34,77.339,6
energy_mu,SARIMAX,2023-05-07,279.5,355.66,76.158,7
energy_mu,SARIMAX,2023-11-12,212.3,285.85,73.55,7
energy_mu,SARIMAX,2023-05-05,314.9,388.2,73.298,5


In [ ]:
from IPython.display import FileLink
for path in sorted((root / 'figures').glob('*.pdf')):
    display(FileLink(str(path.relative_to(root))))